# 05 | Class-subsampling sensitivity and inverse-frequency weighting
**Purpose.** Reproduce Reviewer 1, Comment 2 experiments using a **separate stratified 80/20 split of the FLAME-complete cohort before subsampling**. The original training pool contains 403,444 sources and the common imbalanced test set contains 100,862 sources.

Six CatBoost configurations (1k, 2.5k, 5k, 10k caps, unbalanced, unbalanced weighted) and two Hard Voting configurations (5k and unbalanced) are tested. The same held-out sources are used for all eight, and no training source is present in the held-out test set.

**Important:** Numbers from this notebook are not directly interchangeable with the 7,000-source balanced benchmark.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
flame = read_object(ROOT, "preprocessed.joblib")["flame"]
encoder = LabelEncoder().fit(flame["SpType-ELS"])
train_idx, test_idx = split_indices(flame, cfg)
train_pool, test_pool = flame.iloc[train_idx].copy(), flame.iloc[test_idx].copy()
y_test = encoder.transform(test_pool["SpType-ELS"])
X_test = test_pool[FEATURE_SETS["combined_features"]]
assert set(train_pool["Source"]).isdisjoint(set(test_pool["Source"]))
assert test_pool["Source"].is_unique
print("Training pool:", len(train_pool), "| Common held-out test:", len(test_pool))
display(pd.DataFrame({"Training pool": train_pool["SpType-ELS"].value_counts(), "Test set": test_pool["SpType-ELS"].value_counts()}).reindex(cfg["spectral_classes"]).fillna(0).astype(int))

## Six CatBoost training conditions
Balanced subsampling is applied only within the training pool. Inverse-frequency weights are computed with `sklearn.utils.class_weight.compute_sample_weight(class_weight="balanced")` and passed through the Pipeline as `model__sample_weight`. This is **not** CatBoost's `auto_class_weights` feature.

In [ ]:
settings = [(cap, False) for cap in cfg["sensitivity_caps"]] + [(None, False), (None, True)]
results, predictions, fitted_catboost, training_sets, weights_table = [], {}, {}, {}, []
for cap, weighted in settings:
    name = f"{cap if cap is not None else 'Unbalanced'}_{'weighted' if weighted else 'unweighted'}"
    subset = train_pool.copy() if cap is None else balanced_subsample(train_pool, cap, cfg["random_state"])
    X_train = subset[FEATURE_SETS["combined_features"]]
    y_train = encoder.transform(subset["SpType-ELS"])
    pipe = make_base_pipeline("CatBoost", cfg)
    if weighted:
        weights = compute_sample_weight(class_weight="balanced", y=y_train)
        pipe.fit(X_train, y_train, model__sample_weight=weights)
        for class_index, count in zip(*np.unique(y_train, return_counts=True)):
            weights_table.append({"Class": encoder.classes_[class_index], "Training samples": count, "Sample weight": len(y_train) / (len(encoder.classes_) * count)})
    else:
        pipe.fit(X_train, y_train)
    prediction = np.asarray(pipe.predict(X_test)).ravel().astype(int)
    assert len(prediction) == len(y_test) and set(subset["Source"]).isdisjoint(set(test_pool["Source"]))
    results.append({"Setting": name, "Class cap": cap if cap is not None else "Unbalanced", "Weighted": weighted, "Training samples": len(subset), **metric_dict(y_test, prediction)})
    predictions[name] = prediction
    fitted_catboost[name] = pipe
    training_sets[name] = subset
    print(f"{name}: train={len(subset):,}, test={len(y_test):,}, macro-F1={results[-1]['f1_macro']:.4f}")
sensitivity_table = pd.DataFrame(results)
save_table(sensitivity_table, ROOT, "balancing_catboost.csv")
save_table(pd.DataFrame(weights_table), ROOT, "catboost_balanced_sample_weights.csv")
display(sensitivity_table.round(4)); display(pd.DataFrame(weights_table).round(5))

## Compare the hard-voting ensemble
The CatBoost member reuses its already-fitted model from the matching sensitivity condition. MLP and LightGBM are trained on the **same** corresponding training subset, then evaluated on the fixed held-out test set.

In [ ]:
ensemble_rows, ensemble_preds = [], {}
if cfg["run_balancing_ensemble"]:
    for name in (f"{cfg['class_cap']}_unweighted", "Unbalanced_unweighted"):
        subset = training_sets[name]
        X_train = subset[FEATURE_SETS["combined_features"]]
        y_train = encoder.transform(subset["SpType-ELS"])
        pred, prob = [], []
        for model_name in BASE_MODELS:
            pipe = fitted_catboost[name] if model_name == "CatBoost" else make_base_pipeline(model_name, cfg)
            if model_name != "CatBoost":
                pipe.fit(X_train, y_train)
            pred.append(np.asarray(pipe.predict(X_test)).ravel().astype(int))
            prob.append(pipe.predict_proba(X_test))
        vote, vote_fraction = hard_voting_from_arrays(pred, prob, len(encoder.classes_))
        ensemble_rows.append({"Training setting": name, "Model": "Hard Voting", "Training samples": len(subset), **metric_dict(y_test, vote)})
        ensemble_preds[name] = vote
    ensemble_table = pd.DataFrame(ensemble_rows)
    save_table(ensemble_table, ROOT, "balancing_hard_voting.csv")
    display(ensemble_table.round(4))
else:
    print("Hard-voting sensitivity disabled in config/experiment.json")

## Class-wise recall and effect of weighting
The manuscript presents the recall plot, which highlights that weighting restores sensitivity to the less frequent O-class. The 10,000-source cap is not perfectly balanced because the original sensitivity train pool contains only 6,532 O-class sources.

In [ ]:
selected = [f"{cfg['class_cap']}_unweighted", "Unbalanced_unweighted", "Unbalanced_weighted"]
classwise_rows = []
for name in selected:
    report = classification_report(y_test, predictions[name], labels=np.arange(len(encoder.classes_)), target_names=encoder.classes_, output_dict=True, zero_division=0)
    for cls in encoder.classes_:
        classwise_rows.append({"Setting": name, "Class": cls, "Precision": report[cls]["precision"], "Recall": report[cls]["recall"], "F1": report[cls]["f1-score"], "Support": report[cls]["support"]})
classwise_table = pd.DataFrame(classwise_rows)
save_table(classwise_table, ROOT, "balancing_classwise_recall.csv")
FIG = output_dir(ROOT) / "figures"; FIG.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=classwise_table, x="Class", y="Recall", hue="Setting", ax=ax)
ax.set(title="Class recall under different training distributions", xlabel="Spectral class", ylabel="Class-wise recall", ylim=(0.6, 1.0))
fig.tight_layout(); fig.savefig(FIG / "class_recall_balancing.png"); plt.show()
cap_table = sensitivity_table[sensitivity_table["Class cap"] != "Unbalanced"].copy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cap_table["Class cap"].astype(int), cap_table["f1_macro"], marker="o", linewidth=2)
ax.set(title="Sensitivity to class subsampling — CatBoost", xlabel="Maximum training sources per class", ylabel="Macro-F1")
ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(FIG / "subsampling_sensitivity.png"); plt.show()
save_object({"train_sources": train_pool["Source"].to_numpy(), "test_sources": test_pool["Source"].to_numpy(), "predictions": predictions, "ensemble_predictions": ensemble_preds}, ROOT, "balancing_record.joblib")
print("Saved 6 CatBoost settings and", len(ensemble_rows), "Hard Voting settings.")